# CS535 Project 3 — DeepNeoAG-Lite: Protein Language Models + Multi-window 1D CNN

This notebook is a teaching bridge to the published DeepNeoAG framework.

## Published DeepNeoAG idea
`peptide sequence → pretrained PLM features → multi-window scanning CNN → neoantigen prediction`

The public repository reports:
- 32,877 initial sequences,
- <30% similarity filtering,
- peptide length <35,
- 7,012 final sequences,
- 302 neoantigens and 6,710 other peptides,
- 5,608 train / 1,404 test.

The full repository supports ProtTrans, TAPE, and ESM features. Its example uses `esm2_t33_650M_UR50D`.

## Classroom design
To make the workflow practical on a Colab T4, this notebook first uses a smaller ESM-2 checkpoint (`esm2_t6_8M_UR50D`) and a lightweight MCNN.

For exact paper reproduction, switch to the public repository workflow and published feature files.

In [ ]:
!pip -q install fair-esm scikit-learn pandas

In [ ]:
import os, re, glob, random, time
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, matthews_corrcoef, confusion_matrix
from sklearn.model_selection import train_test_split
import esm

SEED=535
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
if torch.cuda.is_available():torch.cuda.manual_seed_all(SEED)
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
FAST_MODE=True
print("Device:",DEVICE)

## 1. Obtain the published DeepNeoAG repository

This cell clones the public repository. If GitHub access is unavailable in your environment, upload the repository ZIP manually.

In [ ]:
import subprocess

REPO=Path("/content/DeepNeoAG")
if not REPO.exists():
    subprocess.run(
        ["git","clone","-q","https://github.com/B1607/DeepNeoAG.git",str(REPO)],
        check=True
    )
print("repo exists:",REPO.exists())
if REPO.exists():
    print("top-level:",[p.name for p in REPO.iterdir()])


## 2. Discover peptide FASTA files

The original repository may contain multiple FASTA files and feature folders. This helper scans all FASTA-like files and prints them before any modeling decision is made.

For a publication-quality reproduction, use the repository's documented train/test files exactly rather than reconstructing labels heuristically.

In [ ]:
def parse_fasta(path):
    records=[];name=None;seq=[]
    with open(path,encoding="utf-8",errors="ignore") as f:
        for line in f:
            line=line.strip()
            if not line: continue
            if line.startswith(">"):
                if name is not None: records.append((name,"".join(seq)))
                name=line[1:];seq=[]
            else: seq.append(line)
    if name is not None: records.append((name,"".join(seq)))
    return records

fasta_files=[]
if REPO.exists():
    for pat in ["*.fa","*.fasta","*.faa","*.fas"]:
        fasta_files += list(REPO.rglob(pat))
print("FASTA files:",len(fasta_files))
for p in fasta_files[:50]: print(p.relative_to(REPO))

## 3. Two data modes

### Mode A — Recommended for exact reproduction
Use the repository's published train/test peptide files and labels.

### Mode B — Classroom demonstration
Upload or create a CSV with two columns:
- `sequence`
- `label` (1 = neoantigen, 0 = other)

The notebook below automatically looks for a CSV first. If none is available, it attempts to infer labels from FASTA path/header keywords such as `neo`, `positive`, `other`, or `negative`. Always inspect the counts before training.

In [ ]:
def discover_labeled_sequences(repo):
    # 1) Prefer CSV/TSV files with obvious sequence + label columns.
    tables=list(repo.rglob("*.csv"))+list(repo.rglob("*.tsv"))
    for p in tables:
        try:
            df=pd.read_csv(p,sep="\t" if p.suffix==".tsv" else ",")
            low={c.lower():c for c in df.columns}
            seq_col=next((low[k] for k in low if k in ["sequence","seq","peptide","epitope"]),None)
            lab_col=next((low[k] for k in low if k in ["label","class","target","y"]),None)
            if seq_col and lab_col:
                out=df[[seq_col,lab_col]].copy()
                out.columns=["sequence","label"]
                return out,p
        except Exception:
            pass

    # 2) Heuristic FASTA fallback for teaching only.
    rows=[]
    for p in fasta_files:
        text=str(p).lower()
        label=None
        if any(k in text for k in ["neo","positive","pos"]): label=1
        if any(k in text for k in ["other","negative","neg"]): label=0
        if label is None: continue
        for header,seq in parse_fasta(p):
            s=re.sub("[^ACDEFGHIKLMNPQRSTVWY]","",seq.upper())
            if 1<=len(s)<35:
                rows.append((s,label,str(p.relative_to(repo))))
    if rows:
        return pd.DataFrame(rows,columns=["sequence","label","source"]),Path("heuristic_fasta")
    return None,None

df,source = discover_labeled_sequences(REPO) if REPO.exists() else (None,None)
print("source:",source)
if df is not None:
    df["label"]=pd.to_numeric(df["label"],errors="coerce")
    df=df.dropna(subset=["sequence","label"])
    df=df[df["label"].isin([0,1])].copy()
    df["label"]=df["label"].astype(int)
    display(df.head())
    print(df["label"].value_counts())
else:
    print("No automatically labeled table was found.")
    print("Upload a CSV named /content/deepneoag_teaching.csv with columns sequence,label.")

In [ ]:
# Optional upload fallback.
if df is None:
    from google.colab import files
    print("Upload a sequence,label CSV if needed.")
    # uploaded=files.upload()
    # df=pd.read_csv(next(iter(uploaded)))

## 4. If necessary, build a small teaching dataset

Do **not** use synthetic labels for scientific conclusions. The cell below exists only so the architecture can still be demonstrated if the repository format changes.

For a graded research report, students must use the published DeepNeoAG data or another documented biological dataset.

In [ ]:
if df is None:
    # Tiny architecture-only demo.
    pos=["SLYNTVATL","KVAELVHFL","YLEPGPVTA","GLCTLVAML","ILAKFLHWL"]*20
    neg=["AAAAAAAAA","GGGGGGGGG","VVVVVVVVV","SSSSSSSSS","TTTTTTTTT"]*100
    df=pd.DataFrame({"sequence":pos+neg,"label":[1]*len(pos)+[0]*len(neg)})
    print("WARNING: using architecture-only toy labels.")
print(df.shape,df["label"].value_counts().to_dict())

## 5. Train / validation / test

When exact published splits are available, use them. The fallback below uses a stratified split only for classroom demonstration.

In [ ]:
train_df,temp_df=train_test_split(df,test_size=0.3,stratify=df["label"],random_state=SEED)
val_df,test_df=train_test_split(temp_df,test_size=0.5,stratify=temp_df["label"],random_state=SEED)
if FAST_MODE:
    # retain all positives and sample negatives for manageable teaching runtime
    def cap(d,nneg):
        p=d[d.label==1]
        n=d[d.label==0].sample(min(nneg,(d.label==0).sum()),random_state=SEED)
        return pd.concat([p,n]).sample(frac=1,random_state=SEED).reset_index(drop=True)
    train_df=cap(train_df,1200);val_df=cap(val_df,400);test_df=cap(test_df,400)
print(train_df.label.value_counts(),val_df.label.value_counts(),test_df.label.value_counts())

# Part I — One-hot peptide baseline

This makes the protein task look exactly like the earlier text CNN, except the token vocabulary has 20 amino acids.

In [ ]:
AA="ACDEFGHIKLMNPQRSTVWY"
aa2i={a:i for i,a in enumerate(AA)}
MAX_LEN=35

def onehot(seq):
    x=torch.zeros(MAX_LEN,20)
    for i,a in enumerate(seq[:MAX_LEN]):
        if a in aa2i:x[i,aa2i[a]]=1
    return x

class PepOnehotDS(Dataset):
    def __init__(self,d):self.d=d.reset_index(drop=True)
    def __len__(self):return len(self.d)
    def __getitem__(self,i):
        r=self.d.iloc[i]
        return onehot(r.sequence),torch.tensor(float(r.label))

def make_oh_loader(d,bs=64,shuffle=False):
    return DataLoader(PepOnehotDS(d),batch_size=bs,shuffle=shuffle)

oh_tr=make_oh_loader(train_df,64,True);oh_va=make_oh_loader(val_df);oh_te=make_oh_loader(test_df)

In [ ]:
class OneHotMCNN(nn.Module):
    def __init__(self,windows=(2,3,4,5,8),filters=64,dropout=0.3):
        super().__init__()
        self.convs=nn.ModuleList([nn.Conv1d(20,filters,k) for k in windows])
        self.fc=nn.Linear(filters*len(windows),1);self.drop=nn.Dropout(dropout)
    def forward(self,x):
        x=x.transpose(1,2)
        feats=[F.adaptive_max_pool1d(F.relu(c(x)),1).squeeze(-1) for c in self.convs]
        return self.fc(self.drop(torch.cat(feats,1))).squeeze(1)

In [ ]:
def metrics(y,p):
    pred=(p>=0.5).astype(int)
    tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
    return {
        "accuracy":accuracy_score(y,pred),
        "precision":precision_score(y,pred,zero_division=0),
        "recall":recall_score(y,pred,zero_division=0),
        "specificity":tn/(tn+fp) if tn+fp else np.nan,
        "f1":f1_score(y,pred,zero_division=0),
        "mcc":matthews_corrcoef(y,pred),
        "roc_auc":roc_auc_score(y,p),
        "pr_auc":average_precision_score(y,p)
    }

def eval_simple(m,loader):
    m.eval();ys=[];ps=[]
    with torch.no_grad():
        for x,y in loader:
            x,y=x.to(DEVICE),y.to(DEVICE)
            p=torch.sigmoid(m(x))
            ys.extend(y.cpu().numpy());ps.extend(p.cpu().numpy())
    return np.array(ys),np.array(ps)

def fit_simple(m,tr,va,epochs=3,lr=1e-3,pos_weight=None):
    opt=torch.optim.AdamW(m.parameters(),lr=lr)
    pw=torch.tensor([pos_weight],device=DEVICE) if pos_weight is not None else None
    lossfn=nn.BCEWithLogitsLoss(pos_weight=pw)
    best=None;best_pr=-1
    for ep in range(epochs):
        m.train()
        for x,y in tr:
            x,y=x.to(DEVICE),y.to(DEVICE)
            opt.zero_grad(set_to_none=True);loss=lossfn(m(x),y);loss.backward();opt.step()
        yy,pp=eval_simple(m,va);met=metrics(yy,pp);print(ep+1,met)
        if met["pr_auc"]>best_pr:
            best_pr=met["pr_auc"];best={k:v.detach().cpu().clone() for k,v in m.state_dict().items()}
    m.load_state_dict(best)

pos_weight=(train_df.label.eq(0).sum()/max(1,train_df.label.eq(1).sum()))
oh_model=OneHotMCNN().to(DEVICE)
fit_simple(oh_model,oh_tr,oh_va,epochs=2 if FAST_MODE else 8,pos_weight=float(pos_weight))

# Part II — ESM-2 contextual residue embeddings

For classroom speed we use `esm2_t6_8M_UR50D`.  
The published repository also supports larger ESM / ProtTrans / TAPE representations.

In [ ]:
esm_model,alphabet=esm.pretrained.esm2_t6_8M_UR50D()
esm_model=esm_model.to(DEVICE).eval()
batch_converter=alphabet.get_batch_converter()
ESM_DIM=320
for p in esm_model.parameters():p.requires_grad=False

In [ ]:
@torch.no_grad()
def esm_embed_sequences(seqs,batch_size=32):
    out=[]
    for start in range(0,len(seqs),batch_size):
        batch=[(str(i),s) for i,s in enumerate(seqs[start:start+batch_size])]
        labels,strs,tokens=batch_converter(batch)
        tokens=tokens.to(DEVICE)
        rep=esm_model(tokens,repr_layers=[6],return_contacts=False)["representations"][6]
        for j,s in enumerate(strs):
            # Remove BOS and keep residue positions only.
            r=rep[j,1:len(s)+1].detach().cpu()
            pad=torch.zeros(MAX_LEN,ESM_DIM)
            pad[:min(MAX_LEN,len(s))]=r[:MAX_LEN]
            out.append(pad)
    return torch.stack(out)

# Cache embeddings in RAM for the teaching subset.
Xtr_esm=esm_embed_sequences(train_df.sequence.tolist())
Xva_esm=esm_embed_sequences(val_df.sequence.tolist())
Xte_esm=esm_embed_sequences(test_df.sequence.tolist())
print(Xtr_esm.shape)

In [ ]:
class TensorPepDS(Dataset):
    def __init__(self,X,y):self.X=X;self.y=torch.tensor(y,dtype=torch.float32)
    def __len__(self):return len(self.y)
    def __getitem__(self,i):return self.X[i],self.y[i]

esm_tr=DataLoader(TensorPepDS(Xtr_esm,train_df.label.values),batch_size=64,shuffle=True)
esm_va=DataLoader(TensorPepDS(Xva_esm,val_df.label.values),batch_size=64)
esm_te=DataLoader(TensorPepDS(Xte_esm,test_df.label.values),batch_size=64)

class ESM_MCNN(nn.Module):
    def __init__(self,dim=ESM_DIM,windows=(2,3,4,5,8),filters=64,dropout=0.3):
        super().__init__()
        self.convs=nn.ModuleList([nn.Conv1d(dim,filters,k) for k in windows])
        self.drop=nn.Dropout(dropout);self.fc=nn.Linear(filters*len(windows),1)
    def forward(self,x):
        x=x.transpose(1,2)
        feats=[F.adaptive_max_pool1d(F.relu(c(x)),1).squeeze(-1) for c in self.convs]
        return self.fc(self.drop(torch.cat(feats,1))).squeeze(1)

esm_mcnn=ESM_MCNN().to(DEVICE)
fit_simple(esm_mcnn,esm_tr,esm_va,epochs=2 if FAST_MODE else 8,pos_weight=float(pos_weight))

## 6. Independent test comparison

In [ ]:
y1,p1=eval_simple(oh_model,oh_te)
y2,p2=eval_simple(esm_mcnn,esm_te)
display(pd.DataFrame([
    {"model":"one-hot + MCNN",**metrics(y1,p1)},
    {"model":"ESM-2 + MCNN",**metrics(y2,p2)}
]))

## 7. Window-size controlled experiment

Keep ESM-2 embeddings fixed and change only the Conv1D window.

In [ ]:
window_rows=[]
for w in [2,3,4,5,8]:
    m=ESM_MCNN(windows=(w,),filters=64).to(DEVICE)
    fit_simple(m,esm_tr,esm_va,epochs=1 if FAST_MODE else 5,pos_weight=float(pos_weight))
    yy,pp=eval_simple(m,esm_va)
    window_rows.append({"window":w,**metrics(yy,pp)})
display(pd.DataFrame(window_rows))

## 8. Interpret class imbalance

Compare unweighted vs weighted loss and observe the trade-off between precision and recall.

In [ ]:
m_unweighted=ESM_MCNN().to(DEVICE)
fit_simple(m_unweighted,esm_tr,esm_va,epochs=1 if FAST_MODE else 5,pos_weight=None)
yu,pu=eval_simple(m_unweighted,esm_te)
display(pd.DataFrame([
    {"loss":"weighted",**metrics(y2,p2)},
    {"loss":"unweighted",**metrics(yu,pu)}
]))

# From DeepNeoAG-Lite to full research reproduction

For a research-grade reproduction:

1. Use the public repository's exact filtered train/test data.
2. Preserve the <30% similarity-control protocol.
3. Use the published feature-generation scripts for ProtTrans / TAPE / ESM.
4. Perform the intended cross-validation and independent testing.
5. Report MCC, PR-AUC, precision, recall, specificity, ROC-AUC, and confusion-matrix counts.
6. Compare single-window and multi-window models under controlled conditions.
7. Analyze false positives and false negatives biologically.
8. Test external cancer cohorts before claiming broad generalization.